In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "None")

CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


In [ ]:
yaml_content = """
path: /content/dataset

train: images/train
val: images/val
test: images/test

names:
  0: UAV
"""

with open("/content/data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml created successfully!")

data.yaml created successfully!


In [ ]:
import zipfile
from pathlib import Path

# REPLACE with the actual path printed above
zip_path = "/content/drive/MyDrive/RATS-AI/dataset.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/")

print("Dataset extracted!")

Dataset extracted!


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil

# Directories
run_dir = Path("/content/runs/rats/yolo26n_50epochs")
backup_dir = Path(
    "/content/drive/MyDrive/RATS-AI/training/yolo26n_50epochs"
)

# Restore checkpoint
(run_dir / "weights").mkdir(parents=True, exist_ok=True)

shutil.copy2(
    backup_dir / "weights/last.pt",
    run_dir / "weights/last.pt"
)

# Restore other files
for filename in ["best.pt", "results.csv", "args.yaml"]:
    source = (
        backup_dir / "weights" / filename
        if filename.endswith(".pt")
        else backup_dir / filename
    )

    destination = (
        run_dir / "weights" / filename
        if filename.endswith(".pt")
        else run_dir / filename
    )

    if source.exists():
        shutil.copy2(source, destination)

# Automatic backup after each saved epoch
def backup_callback(trainer):
    backup_weights = backup_dir / "weights"
    backup_weights.mkdir(parents=True, exist_ok=True)

    for filename in ["last.pt", "best.pt"]:
        source = run_dir / "weights" / filename

        if source.exists():
            temp = backup_weights / (filename + ".tmp")
            shutil.copy2(source, temp)
            temp.replace(backup_weights / filename)

    for filename in ["results.csv", "args.yaml"]:
        source = run_dir / filename

        if source.exists():
            shutil.copy2(source, backup_dir / filename)

    print(f"Epoch {trainer.epoch + 1}: backed up to Google Drive")

# Resume from checkpoint
model26 = YOLO(str(run_dir / "weights/last.pt"))

model26.add_callback("on_model_save", backup_callback)

model26.train(resume=True)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/rats/yolo26n_50epochs/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_50epochs, nbs=64, nm

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fbe01aa3e70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
from ultralytics import YOLO

models = ["yolo11n.pt", "yolo26n.pt"]

for weights in models:
    name = weights.replace(".pt", "")

    model = YOLO(weights)

    model.train(
        data="/content/data.yaml",
        epochs=50,
        imgsz=640,
        batch=8,
        device=0,
        seed=42,
        project="/content/drive/MyDrive/RATS-AI/runs",
        name=name,
        exist_ok=False
    )

In [ ]:
import shutil
from google.colab import files

run_folder = "/content/runs/rats/yolo11_test"

zip_path = shutil.make_archive(
    "/content/yolo11_test",
    "zip",
    root_dir=run_folder
)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

save_dir = Path("/content/drive/MyDrive/RATS-AI/training")
save_dir.mkdir(parents=True, exist_ok=True)

print("Save directory:", save_dir)

Save directory: /content/drive/MyDrive/RATS-AI/training


In [ ]:
from pathlib import Path

print("Training images:", len(list(Path("/content/dataset/images/train").glob("*.jpg"))))
print("Validation images:", len(list(Path("/content/dataset/images/val").glob("*.jpg"))))
print("data.yaml exists:", Path("/content/data.yaml").exists())

Training images: 0
Validation images: 0
data.yaml exists: True
